# Atlas GasNet E3 — Colab execution evidence

This notebook repeats the **already fixed two-candidate, seed-42 protocol** on
the full E3 dataset. It does not extend the search using previously observed
final-test errors. The retained local E3 run remains a separate artifact.
This is a reproducibility run, not a new independent confirmation dataset.

Execute cells in order. CPU is sufficient; no paid accelerator is requested.
The original E2 report and its results remain unchanged. The broader planned
eight-configuration/five-seed study is outside this reproduction.


In [ ]:
# @title
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time
from IPython.display import HTML, Image, display

ROOT = Path("/content/atlas-gasnet-e3-evidence")
CODE_REVISION = "28daec0f22501d1bd4c0ed3f84c243fdb2dbc4d3"
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/brianUtn98/atlas-gasnet.git", str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", CODE_REVISION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(ROOT / "training/requirements-symbolic.txt")], check=True)
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / "models/matplotlib")

def run_python(script):
    result = subprocess.run([sys.executable, "-u", "-c", script], cwd=ROOT,
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
    result.check_returncode()

print("Pinned code:", CODE_REVISION)
print("Colab evidence run — fixed CPU protocol; all fitting uses train/validation only.")


## 1. Verified input data and experimental configuration

Upload `e3-inputs.zip`, containing only the four original E3 manifests and their
listed Parquet parts. Hash validation and preparation run before any model fits.
Final partitions are integrity-checked without consulting prediction errors.


In [ ]:
# @title
from google.colab import files
from zipfile import ZipFile

INPUT_SHA256 = "51e64ad07fa7d77b0562de436d42eeaa559ab2e6dd20796c988a27cc8e007f32"
input_path = Path("/content/e3-inputs.zip")
if not input_path.exists():
    uploaded = files.upload()
    if "e3-inputs.zip" not in uploaded:
        raise ValueError("Upload the verified e3-inputs.zip archive")
    input_path.write_bytes(uploaded["e3-inputs.zip"])
if hashlib.sha256(input_path.read_bytes()).hexdigest() != INPUT_SHA256:
    raise ValueError("Input archive SHA-256 mismatch")
source = ROOT / "data/extractions/e3-v1"
if not source.exists():
    with ZipFile(input_path) as archive:
        for member in archive.infolist():
            if not (source / member.filename).resolve().is_relative_to(source.resolve()):
                raise ValueError("Invalid archive path")
        archive.extractall(source)
run_python("""
from pathlib import Path
from training.bundles import prepare, load_prepared
from training.prepare_dataset import read_json
prepared = Path("models/e3-prepared")
if prepared.exists():
    load_prepared(prepared)
else:
    prepare(Path("data/extractions/e3-v1"), prepared, "e3")
p = read_json(prepared / "provenance.json")
assert {s: v["rows"] for s, v in p["splits"].items()} == {"train":155435,"validation":39111,"test":59504,"unseen-contract":145124}
for split, data in p["splits"].items():
    print(split, "rows=", data["rows"], "bounds=", data["bounds"])
print("Ordered features:", len(read_json(prepared / "preprocessing.json")["feature_names"]))
print("Preprocessing fitted only on train. No model has been trained.")
""")
display(HTML("""<h3>Fixed experiment configuration</h3><table>
<tr><th>Estimator</th><th>Configuration</th></tr>
<tr><td>MLP</td><td>36–64–32–1; ReLU; Adam 0.001; batch 64; maximum 500 epochs; patience 25; seed 42; CPU/one thread</td></tr>
<tr><td>Genetic Programming</td><td>500 individuals; 20 generations; tournament 20; initial depth 2–4; parsimony 0.001 vs 0.01; seed 42; one worker</td></tr>
<tr><td>Selector median</td><td>Per-function training median, global training median fallback</td></tr>
</table>"""))


## 2. Fit on full training data and select using validation

MLP restores its best-validation-MSE epoch. The symbolic candidate with lower
validation MAE wins; candidate order breaks ties. No final rows are deserialized
by fitting or candidate selection. A new Python process avoids cached libraries.


In [ ]:
# @title
started = time.monotonic()
run_python("""
from pathlib import Path
from training.bundles import train_bundle, load_run
from training.prepare_dataset import read_json
from training.symbolic_experiment import configurations
run = Path("models/e3-colab-run")
if run.exists():
    load_run(run)
    print("Existing complete reproduction run retained; no overwrite.")
else:
    train_bundle(Path("models/e3-prepared"), run,
        symbolic_configs=configurations(read_json(Path("config/training.e3.symbolic.json"))))
print("Training complete. Final-test scores remain closed in this run.")
""")
print("Training cell wall seconds:", round(time.monotonic() - started, 3))


## 3. Learning curves and validation comparison

Training losses have different objectives: normalized MSE for MLP, normalized
MAE for GP. Their gas-unit validation metrics provide the shared comparison.


In [ ]:
# @title
run_python("""
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from training.evaluation import evaluate
from training.prepare_dataset import read_json
run = Path("models/e3-colab-run")
evaluate(run, Path("models/e3-prepared"), validation_only=True,
         verification=(run / "validation").exists())
h = read_json(run / "history.json")
s = read_json(run / "symbolic-history.json")
fig, ax = plt.subplots(figsize=(9,4.5), layout="constrained")
for field in ("train", "validation"):
    ax.plot([e["epoch"] for e in h["epochs"]], [e[field] for e in h["epochs"]], label=field)
ax.axvline(h["best_epoch"], linestyle="--", color="black", label="selected epoch")
ax.set(xlabel="Epoch", ylabel="Normalized MSE", title="E3 Colab — MLP learning curves")
ax.legend(); fig.savefig(run / "mlp-learning.png", dpi=150); plt.close(fig)
fig, ax = plt.subplots(figsize=(9,4.5), layout="constrained")
for candidate in s["candidates"]:
    g = candidate["generations"]
    ax.plot(g["generation"], g["best_fitness"], label=f"candidate {candidate['candidate']}")
ax.set(xlabel="Generation (zero-based)", ylabel="Best normalized training MAE", title="E3 Colab — symbolic evolution")
ax.legend(); fig.savefig(run / "symbolic-learning.png", dpi=150); plt.close(fig)
v = read_json(run / "validation/metrics.json")["validation_diagnostics"]["estimators"]
table = pd.DataFrame([{ "Estimator": k, **x["aggregate"]} for k,x in v.items()])
(run / "validation-table.html").write_text(table.to_html(index=False, float_format=lambda v:f"{v:,.4f}"))
print("MLP epochs:",len(h["epochs"]),"selected epoch:",h["best_epoch"],"stop:",h["stop_reason"])
print("Selected symbolic candidate:",s["selected_candidate"])
for candidate in s["candidates"]:
    print("Candidate",candidate["candidate"],"validation MAE:",candidate["validation_mae_gas"],
          "nodes:",candidate["program_length"],"depth:",candidate["program_depth"])
""")
RUN = ROOT / "models/e3-colab-run"
display(HTML((RUN / "validation-table.html").read_text()))
display(Image(filename=str(RUN / "mlp-learning.png")))
display(Image(filename=str(RUN / "symbolic-learning.png")))


## 4. Record model freeze before final evaluation

This fixed-protocol reproduction performs no new hyperparameter development.
The selected weights, formula and preprocessing are bound by hashes before
computing either final score. Final E3 results were already public from the
separate local run; this execution is explicitly labelled a reproduction.


In [ ]:
# @title
run_python("""
from pathlib import Path
from datetime import datetime, timezone
from training.bundles import load_run, verify
from training.prepare_dataset import read_json, write_json, digest
run = Path("models/e3-colab-run")
*_, manifest = load_run(run)
freeze = run / "model-freeze.json"
if not freeze.exists():
    assert not (run / "evaluation").exists()
    write_json(freeze, {"frozen_at_utc":datetime.now(timezone.utc).isoformat(),
        "protocol":"fixed E3 reproduction; no new tuning", "prior_local_scores_public":True,
        "current_run_final_scores_consulted":False,
        "run_manifest_sha256":digest(run / "manifest.json"),
        "prepared_manifest_sha256":digest(Path("models/e3-prepared/manifest.json")),
        "artifacts":manifest["artifacts"]})
assert read_json(freeze)["run_manifest_sha256"] == digest(run / "manifest.json")
print("Model freeze verified:",read_json(freeze))
""")


## 5. Temporal and unseen-USDT final results

All three estimators use identical row identities. Report temporal and USDT
results separately. The results below cannot justify further E3 tuning.


In [ ]:
# @title
run_python("""
from pathlib import Path
import pandas as pd
from training.evaluation import evaluate
from training.prepare_dataset import read_json
run = Path("models/e3-colab-run")
assert (run / "model-freeze.json").exists()
evaluate(run, Path("models/e3-prepared"), verification=(run / "evaluation").exists())
report = read_json(run / "evaluation/metrics.json")
records = []
for split, section in report["final_tests"].items():
    for estimator, summary in section["estimators"].items():
        records.append({"Partition":split,"Estimator":estimator,**summary["aggregate"]})
table = pd.DataFrame(records)
(run / "final-table.html").write_text(table.to_html(index=False, float_format=lambda v:f"{v:,.4f}"))
print("Separate final evaluations complete; no refitting or parameter changes.")
""")
display(HTML((RUN / "final-table.html").read_text()))
display(Image(filename=str(RUN / "evaluation/test.png")))
display(Image(filename=str(RUN / "evaluation/unseen-contract.png")))


## 6. Validation diagnostics and identified cases

Grouped validation metrics and error quantiles support the report's analysis.
Cases are the three largest MLP validation absolute errors, selected
deterministically by residual magnitude; they do not summarize all observations.


In [ ]:
# @title
run_python("""
from pathlib import Path
import pandas as pd
from training.prepare_dataset import read_json
run = Path("models/e3-colab-run")
section = read_json(run / "validation/metrics.json")["validation_diagnostics"]
grouped = []
for name, model in section["estimators"].items():
    for selector, result in model["functionSelector"].items():
        grouped.append({"Estimator":name,"Selector":selector,"Rows":result["count"],"MAE gas":result["mae"],"RMSE gas":result["rmse"],"R2":result["r2"]})
pd.DataFrame(grouped).to_html(run / "groups-table.html", index=False, float_format=lambda v:f"{v:,.3f}")
frame = pd.read_csv(run / "validation/validation.csv")
diagnostics = []
for name in ("mlp","symbolic","selector_median"):
    residual = frame[f"{name}_residual"]
    diagnostics.append({"Estimator":name,"p50 abs gas":residual.abs().quantile(.5),
        "p90 abs gas":residual.abs().quantile(.9),"p95 abs gas":residual.abs().quantile(.95),
        "max abs gas":residual.abs().max(),"underestimates":int((residual>0).sum()),
        "negative predictions":int((frame[f"{name}_prediction"]<0).sum())})
pd.DataFrame(diagnostics).to_html(run / "diagnostics-table.html",index=False,float_format=lambda v:f"{v:,.3f}")
cases = frame.loc[frame["mlp_residual"].abs().nlargest(3).index,
    ["row_id","functionSelector","actual_gasUsed","mlp_prediction","symbolic_prediction","mlp_residual"]]
cases.to_html(run / "cases-table.html",index=False,float_format=lambda v:f"{v:,.3f}")
print("Display-only symbolic formula:",read_json(run / "symbolic.json")["expression_display_only"])
""")
display(HTML((RUN / "groups-table.html").read_text()))
display(HTML((RUN / "diagnostics-table.html").read_text()))
display(HTML((RUN / "cases-table.html").read_text()))


## 7. Reproduction verification and durable export

Verify unchanged evaluation, export the complete run, reload it in a fresh Python
process and verify original predictions. Download the run and this executed
notebook before the temporary Colab runtime expires.


In [ ]:
# @title
run_python("""
from pathlib import Path
import shutil
from zipfile import ZipFile
from training.bundles import load_run, predict_rows
from training.evaluation import evaluate
from training.prepare_dataset import read_json, digest
run = Path("models/e3-colab-run")
prepared = Path("models/e3-prepared")
evaluate(run, prepared, verification=True)
archive = Path(shutil.make_archive("/content/atlas-gasnet-e3-colab-results", "zip", run.parent, run.name))
restored = Path("models/e3-colab-reloaded")
if not restored.exists():
    with ZipFile(archive) as bundle:
        bundle.extractall(restored)
restored_run = restored / run.name
load_run(restored_run)
sample = read_json(prepared / "validation.json")[:3]
assert predict_rows(restored_run,sample) == predict_rows(run,sample)
print("PASS: final evaluation reproduces exactly")
print("PASS: ZIP reload and exact predictions for all three estimators")
print("Archive SHA-256:",digest(archive))
print("Archive bytes:",archive.stat().st_size)
print("Runtime:",read_json(run / "config.json")["runtime"])
""")
from google.colab import files
files.download("/content/atlas-gasnet-e3-colab-results.zip")


In [ ]:
# @title
def evidence(section):
    run_python(r"""
from pathlib import Path
import pandas as pd
from training.prepare_dataset import read_json, digest
from training.bundles import load_run, verify
run = Path("models/e3-colab-run")
section = __SECTION__
print("ATLAS GASNET E3 — ACTUAL COLAB EXECUTION")
if section == "data":
    p = read_json(run / "provenance.json")
    for split, x in p["splits"].items():
        print(f"{split:18} {x['rows']:>9,} rows  blocks {x['bounds']}")
    print("Features: 36 | Train-only preprocessing | Code:", read_json(run / "config.json")["runtime"]["code_revision"])
    print("MLP: 36-64-32-1; Adam 0.001; batch 64; max 500; patience 25; seed 42; CPU/1 thread")
    print("GP: population 500; generations 20; tournament 20; initial depth 2-4; parsimony .001/.01; seed 42")
elif section == "validation":
    report = read_json(run / "validation/metrics.json")["validation_diagnostics"]
    rows = [{"Estimator":k,"Rows":x["aggregate"]["count"],"MAE gas":x["aggregate"]["mae"],"RMSE gas":x["aggregate"]["rmse"],"R2":x["aggregate"]["r2"]} for k,x in report["estimators"].items()]
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v:f"{v:,.4f}"))
    h=read_json(run / "history.json"); s=read_json(run / "symbolic-history.json")
    print("MLP: epochs",len(h["epochs"]),"selected epoch",h["best_epoch"],"stop",h["stop_reason"])
    for c in s["candidates"]:
        print("GP candidate",c["candidate"],"validation MAE",round(c["validation_mae_gas"],3),"nodes",c["program_length"],"depth",c["program_depth"])
    print("Selected GP candidate:",s["selected_candidate"])
elif section == "final":
    rows=[]
    for split,x in read_json(run / "evaluation/metrics.json")["final_tests"].items():
        for name,v in x["estimators"].items():
            m=v["aggregate"]; rows.append({"Partition":split,"Estimator":name,"Rows":m["count"],"MAE gas":m["mae"],"RMSE gas":m["rmse"],"R2":m["r2"]})
    print(pd.DataFrame(rows).to_string(index=False,float_format=lambda v:f"{v:,.4f}"))
    print("Fixed-model reproduction. No changes based on final-test errors.")
elif section == "groups":
    rows=[]
    for name,x in read_json(run / "validation/metrics.json")["validation_diagnostics"]["estimators"].items():
        for selector,m in x["functionSelector"].items():
            rows.append({"Estimator":name,"Selector":selector,"Rows":m["count"],"MAE gas":m["mae"],"RMSE gas":m["rmse"],"R2":m["r2"]})
    print(pd.DataFrame(rows).to_string(index=False,float_format=lambda v:f"{v:,.3f}"))
elif section == "cases":
    f=pd.read_csv(run / "validation/validation.csv")
    for i,(_,r) in enumerate(f.loc[f["mlp_residual"].abs().nlargest(3).index].iterrows(),1):
        print("CASE",i,"row_id:",r["row_id"])
        print("selector",r["functionSelector"],"actual",r["actual_gasUsed"],"MLP",round(r["mlp_prediction"],3),"GP",round(r["symbolic_prediction"],3),"signed residual",round(r["mlp_residual"],3))
    print("Cases: three largest validation MLP errors; not representative samples.")
elif section == "verification":
    *_,m=load_run(run)
    freeze=read_json(run / "model-freeze.json")
    assert freeze["run_manifest_sha256"] == digest(run / "manifest.json")
    assert freeze["artifacts"] == m["artifacts"]
    verify(run / "evaluation","evaluation")
    p=Path("/content/atlas-gasnet-e3-colab-results.zip")
    print("PASS — complete three-estimator bundle; frozen model hashes unchanged")
    print("PASS — evaluation manifest and all artifact hashes valid")
    print("ZIP bytes:",p.stat().st_size,"SHA-256:",digest(p))
    print("Runtime:",read_json(run / "config.json")["runtime"])
""".replace("__SECTION__",repr(section)))

evidence("validation")


In [ ]:
# @title
evidence("data")

In [ ]:
# @title
evidence("final")

In [ ]:
# @title
evidence("groups")

In [ ]:
# @title
evidence("cases")

In [ ]:
# @title
evidence("verification")

In [ ]:
# @title
run_python(r"""
from pathlib import Path
from zipfile import ZipFile
import shutil
import pandas as pd
from training.bundles import load_run, predict_rows
from training.baseline import labels
from training.evaluation import metrics
from training.prepare_dataset import read_json, write_json, digest
run=Path("models/e3-colab-run")
prepared=Path("models/e3-prepared")
e2_archive=Path("results/colab-2026-10-21/selected-run.zip")
e2_root=Path("models/e2-reference")
if not e2_root.exists():
    with ZipFile(e2_archive) as archive:
        for member in archive.infolist():
            assert (e2_root/member.filename).resolve().is_relative_to(e2_root.resolve())
        archive.extractall(e2_root)
e2=e2_root/"poc-run"
load_run(e2)
report=read_json(run/"evaluation/metrics.json")
records=[]
for split,section in {"validation":report["validation_diagnostics"],**report["final_tests"]}.items():
    rows=read_json(prepared/f"{split}.json")
    actual=labels(rows)
    old=predict_rows(e2,rows)["mlp"]
    records.append({"Partition":split,"Estimator":"E2 retained MLP",**metrics(actual,old)})
    for name,model in section["estimators"].items():
        records.append({"Partition":split,"Estimator":"E3 "+name,**model["aggregate"]})
    pd.DataFrame({"row_id":[f"{r['chainId']}:{r['transactionHash']}" for r in rows],"actual_gasUsed":actual,"e2_mlp_prediction":old}).to_csv(run/f"e2-reference-{split}.csv",index=False)
training=read_json(prepared/"train.json")
current=predict_rows(run,training)
train_metrics={name:metrics(labels(training),current[name]) for name in ("mlp","symbolic","selector_median")}
write_json(run/"supplementary-analysis.json",{"scope":"Fixed E2 external reference plus E3 training diagnostics; no fitting or reselection", "e2_archive_sha256":digest(e2_archive),"same_row_comparison":records,"e3_training_metrics":train_metrics})
print("E3 TRAINING DIAGNOSTICS — restored selected models")
print(pd.DataFrame([{"Estimator":k,**v} for k,v in train_metrics.items()]).drop(columns="r2_undefined_reason").to_string(index=False,float_format=lambda x:f"{x:,.4f}"))
print("FIXED E2 MODEL ON THE SAME E3 ROWS — no retraining")
print(pd.DataFrame(records).drop(columns="r2_undefined_reason").to_string(index=False,float_format=lambda x:f"{x:,.4f}"))
archive=Path(shutil.make_archive("/content/atlas-gasnet-e3-colab-evidence-complete","zip",run.parent,run.name))
print("Complete evidence archive:",archive.name,"bytes:",archive.stat().st_size,"SHA-256:",digest(archive))
""")


In [ ]:
h = json.loads((RUN / "history.json").read_text())
print(f"E3 COLAB - MLP: {len(h['epochs'])} epochs, selected epoch {h['best_epoch']} (validation MSE)")
display(Image(filename=str(RUN / "mlp-learning.png"), width=760))


In [ ]:
s = json.loads((RUN / "symbolic-history.json").read_text())
print(f"E3 COLAB - GP: selected candidate {s['selected_candidate']} by validation MAE")
display(Image(filename=str(RUN / "symbolic-learning.png"), width=760))


## 8. Download the complete evidence archive

Download the complete archive after the supplementary E2 comparison. It includes trained weights and predictions; save this executed notebook separately.


In [ ]:
files.download("/content/atlas-gasnet-e3-colab-evidence-complete.zip")
